In [1]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchaudio
import pandas as pd
import os
from skmultilearn.model_selection import iterative_train_test_split
import torch.optim as optim
from sklearn.metrics import f1_score

In [ ]:
epochs = 20
window_size = 400
hop_length = 160
number_of_mels = 64
model = f"models/model_{window_size}_{hop_length}_{number_of_mels}_{epochs}.pth"

In [ ]:
device = torch.device("cpu")
if torch.cuda.is_available():
   print("Using GPU:", torch.cuda.get_device_name(0))
   device = torch.device("cuda")
elif torch.mps.is_available():
   print("Using MPS device")
   device = torch.device("mps")

In [ ]:
os.makedirs("models", exist_ok=True)

In [2]:
csv_data = pd.read_csv('../dataset/SEP-28k_labels.csv')

In [3]:
csv_data.columns

Index(['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality',
       'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand',
       'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music',
       'NoSpeech'],
      dtype='object')

In [4]:
# Remove music and non-speech data
csv_data = csv_data[(csv_data['Music'] == 0) & (csv_data['NoSpeech'] == 0)]

In [5]:
def multi_label(row):
    return [
        float(row['Prolongation'] > 0),
        float(row['Block'] > 0),
        float((row['SoundRep'] > 0) or (row['WordRep'] > 0)),
        float(row['Interjection'] > 0),
        float(row['NoStutteredWords'] > 0),
    ]

clips_dir = '../dataset/clips/stuttering-clips/clips'

csv_data['label'] = csv_data.apply(multi_label, axis=1)
csv_data['filepath'] = csv_data.apply(
    lambda x: os.path.join(clips_dir, f"{x['Show']}_{x['EpId']}_{x['ClipId']}.wav"), axis=1
)

In [6]:
csv_data.head()

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,WordRep,DifficultToUnderstand,Interjection,NoStutteredWords,NaturalPause,Music,NoSpeech,label,filepath
0,HeStutters,0,0,31900320,31948320,0,0,0,0,0,0,0,0,3,1,0,0,"[0.0, 0.0, 0.0, 0.0, 1.0]",../dataset/clips/stuttering-clips/clips/HeStut...
1,HeStutters,0,1,31977120,32025120,0,0,0,0,0,0,0,0,3,1,0,0,"[0.0, 0.0, 0.0, 0.0, 1.0]",../dataset/clips/stuttering-clips/clips/HeStut...
2,HeStutters,0,2,34809760,34857760,0,0,0,0,0,0,0,0,3,0,0,0,"[0.0, 0.0, 0.0, 0.0, 1.0]",../dataset/clips/stuttering-clips/clips/HeStut...
3,HeStutters,0,3,35200640,35248640,0,0,1,0,0,0,0,0,2,0,0,0,"[1.0, 0.0, 0.0, 0.0, 1.0]",../dataset/clips/stuttering-clips/clips/HeStut...
4,HeStutters,0,4,35721920,35769920,0,0,0,0,0,0,0,0,3,0,0,0,"[0.0, 0.0, 0.0, 0.0, 1.0]",../dataset/clips/stuttering-clips/clips/HeStut...


In [7]:
csv_data.iloc[0].filepath

'../dataset/clips/stuttering-clips/clips/HeStutters_0_0.wav'

In [8]:
X = csv_data['filepath'].values.reshape(-1, 1)
y = np.stack(csv_data['label'].values)  # shape: (num_samples, num_labels)

# Split 70% train, 30% temp
X_train, y_train, X_temp, y_temp = iterative_train_test_split(X, y, test_size=0.3)

# Split temp into val + test 50:50
X_val, y_val, X_test, y_test = iterative_train_test_split(X_temp, y_temp, test_size=0.5)

# Convert back to DataFrame
train_df = pd.DataFrame({'filepath': X_train.flatten(), 'label': list(y_train)})
val_df = pd.DataFrame({'filepath': X_val.flatten(), 'label': list(y_val)})
test_df = pd.DataFrame({'filepath': X_test.flatten(), 'label': list(y_test)})


In [9]:
csv_data.iloc[0].filepath

'../dataset/clips/stuttering-clips/clips/HeStutters_0_0.wav'

In [ ]:
class SEP28kDataset(Dataset):
    def __init__(self, df, sr=16000, n_mels=number_of_mels, max_len=400):
        self.df = df
        self.sr = sr
        self.n_mels = n_mels
        self.max_len = max_len
        self.skipped = 0

        # Torchaudio transforms
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=window_size,
            hop_length=hop_length,
            n_mels=n_mels
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        self.resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=sr)  # in case needed

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row['filepath']

        try:
            waveform, orig_sr = torchaudio.load_with_torchcodec(audio_path, )  # shape: [1, num_samples]

        except RuntimeError as e:
            # raise ValueError(f"Empty waveform: {audio_path}")
            # print(f"Skipping empty file: {audio_path}")
            self.skipped += 1
            return torch.zeros(self.max_len, self.n_mels), torch.zeros(len(row['label']))

        # Resample if needed
        if orig_sr != self.sr:
            waveform = torchaudio.transforms.Resample(orig_sr, self.sr)(waveform)

        # Mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Mel spectrogram
        mel = self.mel_spec(waveform)          # [1, n_mels, time]
        mel_db = self.amplitude_to_db(mel)     # convert to dB

        # Transpose to [time, n_mels]
        mel_db = mel_db.squeeze(0).transpose(0, 1)

        # Pad or truncate to max_len
        if mel_db.shape[0] > self.max_len:
            mel_db = mel_db[:self.max_len, :]
        else:
            pad_len = self.max_len - mel_db.shape[0]
            mel_db = torch.nn.functional.pad(mel_db, (0, 0, 0, pad_len))

        labels = torch.tensor(row['label'], dtype=torch.float32)

        return mel_db, labels


In [11]:
train_dataset = SEP28kDataset(train_df)
val_dataset   = SEP28kDataset(val_df)
test_dataset  = SEP28kDataset(test_df)

In [ ]:
batch_size = 32
pin_memory = torch.cuda.is_available()

train_loader_num_workers = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,           # shuffle for training
    num_workers=train_loader_num_workers,         # parallel data loading, adjust to your CPU cores
    pin_memory=pin_memory         # faster transfer to GPU
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,          # no shuffle for validation
    num_workers=4,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,          # no shuffle for testing
    num_workers=4,
    pin_memory=pin_memory
)


In [15]:
class CNNLSTM(nn.Module):
    def __init__(self, n_mels=64, num_classes=5, hidden_dim=128, num_layers=2):
        super(CNNLSTM, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=(1,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),

            nn.Conv2d(32, 64, kernel_size=(3,3), padding=(1,1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
        )

        self.lstm = nn.LSTM(
            input_size=(n_mels//4)*64,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        # x: (batch, time, n_mels)
        x = x.unsqueeze(1)   # (batch, 1, time, n_mels)
        x = self.conv(x)     # (batch, c, time', mel')

        b, c, t, f = x.shape
        x = x.permute(0,2,1,3).contiguous().view(b, t, c*f)

        out, _ = self.lstm(x)

        # Instead of just last timestep, average across time
        out = out.mean(dim=1)

        out = self.dropout(out)
        out = self.fc(out)   # logits
        return out


In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_labels = 5  # Prolongation, Block, Repetition, Interjection, Fluent

model = CNNLSTM(n_mels=64, num_classes=num_labels, hidden_dim=128, num_layers=2)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()       # multi-label loss
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [17]:
def get_metrics(y_true, y_pred, threshold=0.5):
    y_pred = (torch.sigmoid(y_pred) > threshold).float()
    y_true = y_true.float()

    # Convert to numpy for sklearn metrics
    y_true_np = y_true.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()

    f1 = f1_score(y_true_np, y_pred_np, average='micro')  # micro F1 for multi-label
    return f1

In [ ]:
%%time
for epoch in range(epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        running_f1 = 0

    for mel, labels in train_loader:
        mel, labels = mel.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(mel)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_f1 += get_metrics(labels, outputs)

    train_loss = running_loss / len(train_loader)
    train_f1 = running_f1 / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    val_f1 = 0
    with torch.no_grad():
        for mel, labels in val_loader:
            mel, labels = mel.to(device), labels.to(device)
            outputs = model(mel)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            val_f1 += get_metrics(labels, outputs)

    val_loss /= len(val_loader)
    val_f1 /= len(val_loader)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:f}, Train F1: {train_f1:f} | "
          f"Val Loss: {val_loss:f}, Val F1: {val_f1:f}")

# print(f"Skipped: {train_dataset.skipped + test_dataset.skipped + val_dataset.skipped}")

/Users/skandaprasadk/Documents/mini_project/DADS/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=79, pipe_handle=94)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/anaconda3/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'SEP28kDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>


CPU times: user 147 ms, sys: 75.2 ms, total: 222 ms
Wall time: 4min 17s


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), model)

In [ ]:
model = CNNLSTM(n_mels=64, num_classes=num_labels, hidden_dim=128, num_layers=2)
model.load_state_dict(torch.load(model))
model.eval()

CNNLSTM(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (lstm): LSTM(1024, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=5, bias=True)
)

In [ ]:
exact_correct = 0
total_samples = 0
hamming_correct = 0
total_labels = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)
        predicted = (probs > 0.5).int()
        labels = labels.int()

        # --- Exact match accuracy ---
        match = (predicted == labels).all(dim=1)
        exact_correct += match.sum().item()
        total_samples += labels.size(0)

        # --- Hamming accuracy ---
        hamming_correct += (predicted == labels).sum().item()
        total_labels += labels.numel()

exact_match_acc = exact_correct / total_samples
hamming_acc = hamming_correct / total_labels

print(f"Exact match accuracy: {exact_match_acc:f}")
print(f"Hamming accuracy: {hamming_acc:f}")

Exact match accuracy: 0.27
